Implementation of the Mamba Architecture

In [1]:
import tiktoken
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import urllib.request as req

In [5]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [6]:
try:
    with req.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')
    print("Total Length", len(raw_text))
except:
    print("Text download failed")

Total Length 1115394


In [11]:
corpus = raw_text[:150]
print("Corpus Excerpt: \n", "="*30, "\n", corpus)

Corpus Excerpt: 
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


In [15]:
enc = tiktoken.encoding_for_model("gpt-4o")
token_ids = enc.encode(raw_text)
vocab_size = enc.max_token_value + 1

In [16]:
data_tensor = torch.tensor(token_ids, dtype=torch.long)

Parameters

In [20]:
block_size = 128
batch_size = 16
d_model = 0
d_state = 0
n_layers = 4
learning_rate = 1e-3
max_iters = 1000
eval_interval = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Train/Val Split

In [21]:
n = int(0.9 * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

Batching

In [24]:
def get_batch(split, device):
    data = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Verifying Batching

In [31]:
x, y = get_batch("train", device)
assert x.shape == (batch_size, block_size)
assert y.shape == (batch_size, block_size)